<a href="https://colab.research.google.com/github/reddy7356/-https-github.com-your-username-ep-workflow-resubmission/blob/main/Copy_of_session_05_tools_memory_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Voice Agent From Scratch — Session 5
**Tools and Memory** · Vizuara AI Labs

![Session 5 code pipeline](https://session5-voice-agents-slides.vercel.app/paperbanana/session-05-code-pipeline.png)

### What you're building in Session 5
An **agent loop** that lets the LLM call Python tools and pull useful context from short-term and long-term memory. The Session 4 chatbot becomes a Session 5 agent.

By the end of this notebook you will have:
1. Defined a tool registry with `register`, `dispatch`, and JSON schemas.
2. Registered a real tool: `get_weather(city)`.
3. Called the LLM with the tool attached and let it choose to call the tool.
4. Run the full `LLM -> tool -> LLM` loop in one function.
5. Pointed the loop at a private **sales database** and seen why a bare LLM cannot answer accurately.
6. Added a turn buffer for short-term memory and seen multi-turn behaviour.
7. Added a chromadb vector store for long-term memory.
8. Swapped the agent into the Session 1 scaffold without touching the orchestrator.

> Set your `OPENAI_API_KEY` in Colab Secrets (or as a regular env var) before running. The notebook is structured so each cell builds on the previous one.

## 1 · Install dependencies
`openai` for the LLM with tool calling, `chromadb` for the local vector store, `tiktoken` for counting prompt tokens.

In [ ]:
import sys, subprocess

PKGS = ['openai>=1.40.0', 'chromadb>=0.5.0', 'tiktoken']
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', *PKGS])

import openai, chromadb, tiktoken
print('openai', openai.__version__)
print('chromadb', chromadb.__version__)

openai 2.41.0
chromadb 1.5.9


## 2 · Set your OpenAI API key
In Colab, prefer `userdata` over hard-coding. The cell below falls back to `os.environ` if you set it manually.

In [ ]:
import os

try:
    from google.colab import userdata
    # Assuming the secret in Colab is named 'OPENAI_API_KEY'
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass  # already set, or running outside Colab

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY before continuing.'
print('OPENAI_API_KEY is set.')

OPENAI_API_KEY is set.


## 3 · Define a real Python tool
We start with a deterministic mock `get_weather`. Real code, no network: easy to reason about, easy to test. We will plug a real API in later.

In [ ]:
def get_weather(city: str, units: str = 'c') -> dict:
    """Mock weather lookup. Replace with a real API later."""
    table = {
        'paris':   {'temp_c': 12, 'sky': 'clear'},
        'mumbai':  {'temp_c': 31, 'sky': 'humid'},
        'tokyo':   {'temp_c': 18, 'sky': 'rain'},
    }
    row = table.get(city.lower())
    if row is None:
        return {'error': f'no data for {city}'}
    temp = row['temp_c'] if units == 'c' else round(row['temp_c'] * 9/5 + 32)
    return {'city': city, 'temp': temp, 'units': units, 'sky': row['sky']}

print(get_weather('Paris'))
print(get_weather('Tokyo', units='f'))

{'city': 'Paris', 'temp': 12, 'units': 'c', 'sky': 'clear'}
{'city': 'Tokyo', 'temp': 64, 'units': 'f', 'sky': 'rain'}


## 4 · The tool registry
A tiny registry pattern: one mapping from name to (function, schema). `dispatch` is the only place we ever run a tool, so error handling lives in one place.

In [ ]:
TOOLS = {}

def register(name, fn, schema):
    TOOLS[name] = {'fn': fn, 'schema': schema}

def schemas():
    return [t['schema'] for t in TOOLS.values()]

def dispatch(name, args):
    if name not in TOOLS:
        return {'error': f'unknown tool {name}'}
    try:
        return TOOLS[name]['fn'](**args)
    except Exception as e:
        return {'error': f'{type(e).__name__}: {e}'}

WEATHER_SCHEMA = {
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Look up the current weather for a city.',
        'parameters': {
            'type': 'object',
            'properties': {
                'city':  {'type': 'string', 'description': 'City name, e.g. Paris'},
                'units': {'type': 'string', 'enum': ['c', 'f'], 'description': 'Temperature units'},
            },
            'required': ['city'],
        },
    },
}

register('get_weather', get_weather, WEATHER_SCHEMA)
print('Registered:', list(TOOLS.keys()))

Registered: ['get_weather']


## 5 · One LLM call with tools attached
We pass `tools=schemas()` to the model. If the model picks our tool, it returns a `tool_calls` field instead of a normal message. We do not run anything yet — just inspect the response.

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

messages = [
    {'role': 'system', 'content': 'You are a helpful voice assistant. Be concise.'},
    {'role': 'user',   'content': 'What is the weather in Paris right now?'},
]

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=schemas(),
)
msg = resp.choices[0].message

if msg.tool_calls:
    call = msg.tool_calls[0]
    print('Model wants to call:', call.function.name)
    print('With arguments    :', call.function.arguments)
else:
    print('Model answered directly:', msg.content)

Model wants to call: get_weather
With arguments    : {"city":"Paris"}


## 6 · The full agent loop
`agent_step` calls the LLM, runs every tool the model asked for, appends the results, and loops until the model returns a normal message. The `max_calls` cap prevents runaway tool storms.

In [ ]:
def agent_step(messages, max_calls=4):
    """Run the LLM -> tool -> LLM loop until the model produces a final answer."""
    for _ in range(max_calls):
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=schemas(),
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            messages.append({'role': 'assistant', 'content': msg.content})
            return msg.content

        # Record the assistant message (with tool_calls) so the next turn sees them
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = dispatch(name, args)
            print(f'  [tool] {name}({args}) -> {result}')
            messages.append({
                'role': 'tool',
                'tool_call_id': call.id,
                'content': json.dumps(result),
            })
    return 'I tried but could not finish that.'

messages = [
    {'role': 'system', 'content': 'You are a helpful voice assistant. Be concise.'},
    {'role': 'user',   'content': 'What is the weather in Paris, in Fahrenheit?'},
]
answer = agent_step(messages)
print('\nAgent:', answer)

  [tool] get_weather({'city': 'Paris', 'units': 'f'}) -> {'city': 'Paris', 'temp': 54, 'units': 'f', 'sky': 'clear'}

Agent: The weather in Paris is 54°F and the sky is clear.


## 7 · A real, private tool: query your sales database
The mock weather tool is cute, but the real payoff is data the model has **never seen**. Below we build a tiny **Brew Haven Coffee Co.** dataset (10 products, 6 months, 4 regions) and a `query_sales` tool. Then we ask the same question twice — once with no tools, once with the tool — and compare.

The golden rule: **all aggregation happens in deterministic Python**, never in the model's head. The LLM only decides which tool to call and how to phrase the answer.

> The same dataset ships in the repo as `data/products.csv`, `data/sales.csv`, and `data/brew_haven_sales.xlsx`.

In [ ]:
import random

# id, name, category, unit_price_inr, description
PRODUCTS = [
    ('BH-101', 'Signature Espresso Blend', 'Beans', 650, 'Flagship medium-dark roast, 100% arabica. 250g.'),
    ('BH-102', 'Colombian Single Origin', 'Beans', 720, 'Bright washed Colombian from Huila. 250g.'),
    ('BH-103', 'Dark Roast Decaf', 'Beans', 690, 'Swiss-water decaf dark roast, full body. 250g.'),
    ('BH-104', 'Hazelnut Flavoured Beans', 'Beans', 600, 'Medium roast infused with natural hazelnut. 250g.'),
    ('BH-201', 'Cold Brew Concentrate', 'Beverages', 480, '18-hour cold brew concentrate, 1L, makes ~8 cups.'),
    ('BH-202', 'Vanilla Oat Latte Mix', 'Mixes', 420, 'Instant oat-milk latte powder with real vanilla. 300g.'),
    ('BH-301', 'Ceramic Pour-Over Dripper', 'Equipment', 950, 'Hand-glazed ceramic V-style dripper, size 02.'),
    ('BH-302', 'French Press 1L', 'Equipment', 1350, 'Borosilicate glass French press, brews 8 cups.'),
    ('BH-401', 'Insulated Travel Tumbler', 'Merchandise', 890, 'Double-walled 350ml steel tumbler, 6h hot.'),
    ('BH-501', 'Festive Gift Hamper', 'Bundles', 2200, 'Gift box: blend, cold brew, tumbler, two cups.'),
]
REGIONS = ['North', 'South', 'East', 'West']
MONTHS = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
THIS_MONTH = '2026-06'  # session date is 2026-06-08
BASE_UNITS = {'BH-101': 540, 'BH-102': 410, 'BH-103': 220, 'BH-104': 260, 'BH-201': 380,
              'BH-202': 300, 'BH-301': 150, 'BH-302': 120, 'BH-401': 200, 'BH-501': 90}
REGION_WEIGHT = {'North': 0.30, 'South': 0.28, 'East': 0.18, 'West': 0.24}
MONTH_TREND = {'2026-01': 0.92, '2026-02': 0.95, '2026-03': 1.00, '2026-04': 1.03, '2026-05': 1.08, '2026-06': 1.12}

# Build the same rows as the repo CSV (seeded -> identical numbers).
_rng = random.Random(42)
PRICE = {pid: pr for pid, _, _, pr, _ in PRODUCTS}
SALES = []
for _month in MONTHS:
    for _pid in PRICE:
        for _region in REGIONS:
            _base = BASE_UNITS[_pid] * REGION_WEIGHT[_region] * MONTH_TREND[_month]
            _units = max(int(round(_base * _rng.uniform(0.88, 1.12))), 0)
            SALES.append({'month': _month, 'product_id': _pid, 'region': _region,
                          'units_sold': _units, 'revenue': _units * PRICE[_pid]})

print(f'Brew Haven dataset: {len(PRODUCTS)} products, {len(SALES)} sales rows')
print('Sanity — June 2026 total revenue:',
      sum(r['revenue'] for r in SALES if r['month'] == THIS_MONTH))

Brew Haven dataset: 10 products, 240 sales rows
Sanity — June 2026 total revenue: 2199500


In [ ]:
# --- The deterministic sales tools ---
def query_sales(month=None, region=None, product_id=None, group_by=None, metric='revenue'):
    """Aggregate Brew Haven sales. Returns EXACT units and revenue (INR)."""
    rows = SALES
    if month:      rows = [r for r in rows if r['month'] == month]
    if region:     rows = [r for r in rows if r['region'].lower() == region.lower()]
    if product_id: rows = [r for r in rows if r['product_id'].lower() == product_id.lower()]
    if not rows:   return {'error': 'no rows match those filters'}
    agg = {}
    for r in rows:
        key = (r['product_id'] if group_by == 'product' else
               r['region'] if group_by == 'region' else
               r['month'] if group_by == 'month' else 'total')
        b = agg.setdefault(key, {'units': 0, 'revenue': 0})
        b['units'] += r['units_sold']; b['revenue'] += r['revenue']
    sort_key = 'units' if metric == 'units' else 'revenue'
    results = sorted(({'key': k, **v} for k, v in agg.items()), key=lambda x: x[sort_key], reverse=True)
    total = {'units': sum(r['units'] for r in results), 'revenue': sum(r['revenue'] for r in results)}
    return {'currency': 'INR', 'metric': metric, 'group_by': group_by or 'none', 'results': results, 'total': total}

def get_product_info(query):
    q = (query or '').strip().lower()
    for pid, name, cat, price, desc in PRODUCTS:
        if pid.lower() == q or q in name.lower() or name.lower() in q:
            return {'product_id': pid, 'name': name, 'category': cat, 'unit_price_inr': price, 'description': desc}
    return {'error': f'no product matching {query!r}'}

def _sum_where(dimension, value, month, metric):
    field = 'units_sold' if metric == 'units' else 'revenue'
    rows = SALES
    if month: rows = [r for r in rows if r['month'] == month]
    if dimension == 'region':  rows = [r for r in rows if r['region'].lower() == value.lower()]
    elif dimension == 'product': rows = [r for r in rows if r['product_id'].lower() == value.lower()]
    elif dimension == 'month': rows = [r for r in rows if r['month'] == value]
    return {'rows': len(rows), 'value': sum(r[field] for r in rows)}

def compare_sales(dimension, a, b, month=None, metric='revenue'):
    """Compare two regions, products, or months head-to-head (deterministic)."""
    a_res = _sum_where(dimension, a, None if dimension == 'month' else month, metric)
    b_res = _sum_where(dimension, b, None if dimension == 'month' else month, metric)
    diff = a_res['value'] - b_res['value']
    pct = round(diff / b_res['value'] * 1000) / 10 if b_res['value'] else None
    return {'currency': 'INR', 'metric': metric, 'dimension': dimension,
            'a': {'key': a, 'value': a_res['value']}, 'b': {'key': b, 'value': b_res['value']},
            'difference': diff, 'percent_a_vs_b': pct, 'higher': 'tie' if diff == 0 else (a if diff > 0 else b)}

def sales_growth(from_month, to_month, region=None, product_id=None, metric='revenue'):
    """Change and growth %% in sales between two months (deterministic)."""
    field = 'units_sold' if metric == 'units' else 'revenue'
    def sm(m):
        rows = [r for r in SALES if r['month'] == m]
        if region:     rows = [r for r in rows if r['region'].lower() == region.lower()]
        if product_id: rows = [r for r in rows if r['product_id'].lower() == product_id.lower()]
        return sum(r[field] for r in rows)
    frm, to = sm(from_month), sm(to_month)
    change = to - frm
    pct = round(change / frm * 1000) / 10 if frm else None
    return {'currency': 'INR', 'metric': metric, 'from': {'month': from_month, 'value': frm},
            'to': {'month': to_month, 'value': to}, 'change': change, 'growth_pct': pct}

QUERY_SALES_SCHEMA = {
    'type': 'function',
    'function': {
        'name': 'query_sales',
        'description': ('Aggregate Brew Haven sales; returns exact units and revenue in INR. '
                        'Use for ANY totals/bestseller/regional question. Today is 2026-06-08 so this month = 2026-06.'),
        'parameters': {'type': 'object', 'properties': {
            'month': {'type': 'string', 'description': 'YYYY-MM, e.g. 2026-06. Omit for all months.'},
            'region': {'type': 'string', 'enum': REGIONS, 'description': 'Omit for all regions.'},
            'product_id': {'type': 'string', 'description': 'e.g. BH-101. Omit for all products.'},
            'group_by': {'type': 'string', 'enum': ['product', 'region', 'month'], 'description': 'Break down by this. Omit for a grand total.'},
            'metric': {'type': 'string', 'enum': ['revenue', 'units'], 'description': 'Rank by this. Default revenue.'},
        }},
    },
}
PRODUCT_INFO_SCHEMA = {
    'type': 'function',
    'function': {
        'name': 'get_product_info',
        'description': 'Look up one product description, category, and unit price by name or id.',
        'parameters': {'type': 'object',
            'properties': {'query': {'type': 'string', 'description': "Product name or id, e.g. 'Cold Brew Concentrate' or 'BH-201'."}},
            'required': ['query']},
    },
}
COMPARE_SALES_SCHEMA = {
    'type': 'function',
    'function': {
        'name': 'compare_sales',
        'description': 'Compare two regions, products, or months head-to-head; returns both figures, the difference, the percent difference, and which is higher.',
        'parameters': {'type': 'object', 'properties': {
            'dimension': {'type': 'string', 'enum': ['region', 'product', 'month'], 'description': 'What you are comparing.'},
            'a': {'type': 'string', 'description': "First value, e.g. 'North', 'BH-101', or '2026-05'."},
            'b': {'type': 'string', 'description': 'Second value to compare against.'},
            'month': {'type': 'string', 'description': "Optional YYYY-MM filter (ignored when dimension is 'month')."},
            'metric': {'type': 'string', 'enum': ['revenue', 'units'], 'description': 'Default revenue.'},
        }, 'required': ['dimension', 'a', 'b']},
    },
}
SALES_GROWTH_SCHEMA = {
    'type': 'function',
    'function': {
        'name': 'sales_growth',
        'description': 'Compute the change and growth percentage in sales between two months. Optionally scope to a region or product.',
        'parameters': {'type': 'object', 'properties': {
            'from_month': {'type': 'string', 'description': 'Start month, YYYY-MM, e.g. 2026-01.'},
            'to_month': {'type': 'string', 'description': 'End month, YYYY-MM, e.g. 2026-06.'},
            'region': {'type': 'string', 'enum': REGIONS, 'description': 'Optional region scope.'},
            'product_id': {'type': 'string', 'description': 'Optional product id scope, e.g. BH-201.'},
            'metric': {'type': 'string', 'enum': ['revenue', 'units'], 'description': 'Default revenue.'},
        }, 'required': ['from_month', 'to_month']},
    },
}
register('query_sales', query_sales, QUERY_SALES_SCHEMA)
register('get_product_info', get_product_info, PRODUCT_INFO_SCHEMA)
register('compare_sales', compare_sales, COMPARE_SALES_SCHEMA)
register('sales_growth', sales_growth, SALES_GROWTH_SCHEMA)
print('Registered:', list(TOOLS.keys()))
print('Direct call — total sales this month:', query_sales(month=THIS_MONTH)['total'])
print('Direct call — Jan→Jun growth:', sales_growth('2026-01', '2026-06'))

Registered: ['get_weather', 'query_sales', 'get_product_info', 'compare_sales', 'sales_growth']
Direct call — total sales this month: {'units': 3032, 'revenue': 2199500}
Direct call — Jan→Jun growth: {'currency': 'INR', 'metric': 'revenue', 'from': {'month': '2026-01', 'value': 1783020}, 'to': {'month': '2026-06', 'value': 2199500}, 'change': 416480, 'growth_pct': 23.4}


### The contrast: same question, with and without the tool
With no tools the model cannot see the books — it refuses or, worse, invents a number. With `query_sales` it returns the exact figure, every time. This is the whole reason tools exist.

In [ ]:
QUESTION = 'What were our total sales this month, and which product sold the most?'

# --- WITHOUT tools: the model has no access to the data ---
no_tool_msgs = [
    {'role': 'system', 'content': 'You are the Brew Haven Coffee Co. sales analyst. Today is 2026-06-08. You have no database access.'},
    {'role': 'user', 'content': QUESTION},
]
resp = client.chat.completions.create(model='gpt-4o-mini', messages=no_tool_msgs)
print('WITHOUT TOOLS:\n', resp.choices[0].message.content)

# --- WITH tools: agent_step lets the model call query_sales ---
with_tool_msgs = [
    {'role': 'system', 'content': ('You are the Brew Haven Coffee Co. sales analyst. Today is 2026-06-08 so '
                                   '"this month" means 2026-06. For any figures you MUST call a tool and report '
                                   'its numbers verbatim — never estimate.')},
    {'role': 'user', 'content': QUESTION},
]
print('\nWITH TOOLS:')
print(agent_step(with_tool_msgs))

WITHOUT TOOLS:
 I don't have access to the specific sales data for Brew Haven Coffee Co. Since it is June 2026, the best way for you to find out the total sales for this month and the best-selling product would be to check your sales database or reporting tool. If you need help with analysis or strategies for understanding your sales data, feel free to ask!

WITH TOOLS:
  [tool] query_sales({'month': '2026-06'}) -> {'currency': 'INR', 'metric': 'revenue', 'group_by': 'none', 'results': [{'key': 'total', 'units': 3032, 'revenue': 2199500}], 'total': {'units': 3032, 'revenue': 2199500}}
  [tool] query_sales({'month': '2026-06', 'group_by': 'product', 'metric': 'units'}) -> {'currency': 'INR', 'metric': 'units', 'group_by': 'product', 'results': [{'key': 'BH-101', 'units': 641, 'revenue': 416650}, {'key': 'BH-102', 'units': 437, 'revenue': 314640}, {'key': 'BH-201', 'units': 421, 'revenue': 202080}, {'key': 'BH-202', 'units': 354, 'revenue': 148680}, {'key': 'BH-104', 'units': 302, 'reven

## 8 · Short-term memory: a turn buffer
The agent loop above does not remember anything across turns. We add a turn buffer that keeps the last N messages verbatim, so pronouns and follow-ups resolve correctly.

In [ ]:
class TurnBuffer:
    def __init__(self, system_prompt, max_turns=10):
        self.system_prompt = system_prompt
        self.turns = []          # list of role/content dicts (and tool messages)
        self.max_turns = max_turns

    def messages(self):
        return [{'role': 'system', 'content': self.system_prompt}] + self.turns

    def push(self, *new_msgs):
        self.turns.extend(new_msgs)
        # Drop oldest non-system messages once we exceed the limit
        if len(self.turns) > self.max_turns:
            self.turns = self.turns[-self.max_turns:]

buf = TurnBuffer('You are a helpful voice assistant. Be concise.', max_turns=12)

def chat(user_text):
    buf.push({'role': 'user', 'content': user_text})
    msgs = buf.messages()
    answer = agent_step(msgs)
    # agent_step already appended assistant + tool messages onto msgs.
    # Copy the newly added messages back into the buffer.
    added = msgs[len(buf.messages()):]
    buf.push(*added)
    return answer

print('User:', 'Weather in Tokyo?')
print('Agent:', chat('Weather in Tokyo?'))
print()
print('User:', 'And in Mumbai?')
print('Agent:', chat('And in Mumbai?'))

User: Weather in Tokyo?
  [tool] get_weather({'city': 'Tokyo'}) -> {'city': 'Tokyo', 'temp': 18, 'units': 'c', 'sky': 'rain'}
Agent: The current weather in Tokyo is 18°C with rain.

User: And in Mumbai?
  [tool] get_weather({'city': 'Mumbai'}) -> {'city': 'Mumbai', 'temp': 31, 'units': 'c', 'sky': 'humid'}
Agent: The current weather in Mumbai is 31°C and humid.


## 9 · Compressing older turns with a running summary
When the buffer fills up, we ask the LLM to compress the oldest half into a short summary string and replace those turns. The most recent turns stay verbatim so references still resolve.

In [ ]:
def summarise(turns):
    """Compress a list of turns into one short summary string."""
    flat = '\n'.join(
        f"{m.get('role','?')}: {m.get('content','')}" for m in turns if m.get('content')
    )
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Summarise this voice-agent conversation in 2 short sentences. Keep names, places, and preferences.'},
            {'role': 'user', 'content': flat},
        ],
    )
    return resp.choices[0].message.content.strip()

# Run summariser on what's currently in the buffer
summary = summarise(buf.turns[:max(2, len(buf.turns)//2)])
print('Running summary:\n ', summary)

Running summary:
  The user asked about the weather in Tokyo, which is currently 18°C and rainy.


## 10 · Long-term memory: a tiny vector store
chromadb runs in-process with a local file. We store one note per row, tagged with `user_id`. Retrieval filters by `user_id` so different users do not see each other's notes.

In [ ]:
import chromadb
import os

MEM_DIR = '/content/mem' if os.path.exists('/content') else './mem'
chroma = chromadb.PersistentClient(path=MEM_DIR)
col = chroma.get_or_create_collection('notes')

def remember(text, user_id):
    col.add(documents=[text], metadatas=[{'user_id': user_id}], ids=[f'{user_id}-{abs(hash(text))}'])

def recall(query, user_id, k=3):
    r = col.query(query_texts=[query], n_results=k, where={'user_id': user_id})
    return r['documents'][0] if r['documents'] else []

# Save a few notes for user 'alice'
remember('Alice is vegan and dislikes mushrooms.', user_id='alice')
remember('Alice lives in Mumbai.', user_id='alice')
remember('Alice prefers Piper TTS over ElevenLabs.', user_id='alice')

# Save one note for a different user to check isolation
remember('Bob is in Tokyo.', user_id='bob')

print('Alice recall for "food preferences":')
for hit in recall('food preferences', user_id='alice'):
    print(' -', hit)

print('\nBob recall for "food preferences":')
for hit in recall('food preferences', user_id='bob'):
    print(' -', hit)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Alice recall for "food preferences":
 - Alice is vegan and dislikes mushrooms.
 - Alice prefers Piper TTS over ElevenLabs.
 - Alice lives in Mumbai.

Bob recall for "food preferences":
 - Bob is in Tokyo.


## 11 · Putting it all together: tools + short-term + long-term
One function that assembles the prompt from three memory layers, runs the agent loop, and returns a spoken answer.

In [ ]:
def voice_agent_turn(user_text, user_id='alice'):
    # 1. Pull top-k relevant notes from long-term memory
    hits = recall(user_text, user_id=user_id, k=3)
    memory_block = '\n'.join(f'- {h}' for h in hits) or '(no relevant notes)'

    system_prompt = (
        f'You are a helpful voice assistant for user {user_id}. Be concise.\n'
        f'Known facts about this user:\n{memory_block}'
    )

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_text},
    ]
    return agent_step(messages)

print(voice_agent_turn('What is the weather like where I live?'))
print()
print(voice_agent_turn('Suggest dinner for tonight.'))

  [tool] get_weather({'city': 'Mumbai', 'units': 'c'}) -> {'city': 'Mumbai', 'temp': 31, 'units': 'c', 'sky': 'humid'}
The weather in Mumbai is currently 31°C and humid.

How about a vegan Buddha bowl? You can include quinoa or brown rice as the base, topped with roasted veggies (like sweet potatoes and broccoli), chickpeas, avocado, and a tahini or lemon dressing. Just skip any mushrooms!


## 12 · Where the time goes
Compared to a no-tool turn, a tool-using turn pays for two LLM calls plus the tool. Budget accordingly.

In [ ]:
from IPython.display import Image, display
display(Image(url='https://session5-voice-agents-slides.vercel.app/paperbanana/tool-latency-budget.png'))

## 13 · Swap the agent into the Session 1 scaffold
The orchestrator does not change. Only the LLM module is replaced by `agent_step`. The conversation buffer and the vector store hang off it.

In [ ]:
from pathlib import Path
import shutil, sys

ROOT = Path('/content/s5_voice_agent') if Path('/content').exists() else Path.cwd() / 's5_voice_agent'
SRC  = ROOT / 'src'
if ROOT.exists(): shutil.rmtree(ROOT)
SRC.mkdir(parents=True)

# Session 1 ASR / TTS placeholders (unchanged)
(SRC / 'asr.py').write_text('def transcribe(text):\n    return text\n', encoding='utf-8')
(SRC / 'tts.py').write_text('def speak(text):\n    print(f"Agent: {text}")\n', encoding='utf-8')

# Session 5 LLM module: an agent with tools
(SRC / 'llm.py').write_text(
    'from __main__ import agent_step  # reuse the loop we built above\n'
    'def generate_response(transcript, history=None):\n'
    '    msgs = [{"role": "system", "content": "You are a helpful voice assistant. Be concise."}]\n'
    '    if history: msgs.extend(history)\n'
    '    msgs.append({"role": "user", "content": transcript})\n'
    '    return agent_step(msgs)\n',
    encoding="utf-8",
)

# Orchestrator from Session 1
(SRC / 'orchestrator.py').write_text(
    'import asr, llm, tts\n'
    'def run_turn(user_text):\n'
    '    t = asr.transcribe(user_text)\n'
    '    r = llm.generate_response(t)\n'
    '    tts.speak(r)\n'
    '    return r\n',
    encoding="utf-8",
)

if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
for m in ('asr', 'llm', 'tts', 'orchestrator'):
    if m in sys.modules: del sys.modules[m]

from orchestrator import run_turn
print('One full turn through the Session 1 loop, now powered by the Session 5 agent:\n')
run_turn('What is the weather in Tokyo?')

One full turn through the Session 1 loop, now powered by the Session 5 agent:

  [tool] get_weather({'city': 'Tokyo'}) -> {'city': 'Tokyo', 'temp': 18, 'units': 'c', 'sky': 'rain'}
Agent: The weather in Tokyo is 18°C with rain.


'The weather in Tokyo is 18°C with rain.'

## What you have now
| Piece | Today | Replaced in |
| --- | --- | --- |
| ASR | placeholder (Session 2 swaps in faster-whisper) | Session 6 streaming |
| LLM | gpt-4o-mini + tool calling + agent loop | Session 7 (custom routing) |
| Tools | `get_weather` (mock) + `query_sales` over a private dataset; registry pattern | Session 7 (real APIs) |
| Short-term memory | turn buffer (last N messages) | Session 6 streaming history |
| Summary | one-shot LLM compression on overflow | tune cadence per app |
| Long-term memory | chromadb collection, filtered by `user_id` | swap to pgvector at scale |

### Things to try
- Register a second tool: `calculator(expression)` that evaluates a math string.
- Add a `remember(text)` tool so the agent can decide to write to long-term memory itself.
- Set `max_calls=1` in `agent_step` and try a query that genuinely needs two tools — watch the failure mode.
- Add a slow tool (`time.sleep(2)` inside its body) and measure how the turn latency changes.

### Next session
Session 6 turns this whole thing real-time: streaming ASR, streaming LLM tokens, streaming TTS, and barge-in.